In [14]:
def analyze_benchmark_leak(file_path, folder_type, time_col='Hourly', consumption_col='Consumption m3'):
    """
    Benchmark version of the leak detection logic.
    Optimized to return comprehensive reason breakdowns for verification.
    """
    try:
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path, header=0)
        else:
            df = pd.read_excel(file_path, header=0) 
            
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
        
        # Extract Time Features
        df['hour'] = df[time_col].dt.hour
        df['day_of_week'] = df[time_col].dt.dayofweek
        
        # 8-Week Rolling Timeline Windows
        max_date = df[time_col].max()
        four_weeks_ago = max_date - pd.Timedelta(weeks=4)
        eight_weeks_prior = four_weeks_ago - pd.Timedelta(weeks=8)
        
        df['period'] = 'Ignore' 
        df.loc[(df[time_col] >= eight_weeks_prior) & (df[time_col] < four_weeks_ago), 'period'] = 'Historical_Baseline'
        df.loc[df[time_col] >= four_weeks_ago, 'period'] = 'Recent_Evaluation'
        
        # --- Build Baseline Profiles ---
        baseline_data = df[df['period'] == 'Historical_Baseline']
        if baseline_data.empty:
            baseline_data = df[df[time_col] < four_weeks_ago] 
        if baseline_data.empty:
            baseline_data = df 
            
        baseline_profile = baseline_data.groupby(['day_of_week', 'hour'])[consumption_col].median().reset_index()
        baseline_profile.rename(columns={consumption_col: 'baseline_median'}, inplace=True)
        baseline_std = baseline_data.groupby(['day_of_week', 'hour'])[consumption_col].std().reset_index()
        baseline_std.rename(columns={consumption_col: 'baseline_std'}, inplace=True)
        
        df = pd.merge(df, baseline_profile, on=['day_of_week', 'hour'], how='left')
        df = pd.merge(df, baseline_std, on=['day_of_week', 'hour'], how='left')
        
        # --- Compute Structural Deviations ---
        df['abs_deviation'] = df[consumption_col] - df['baseline_median']
        global_median = df[consumption_col].median()
        std_safety_floor = max(0.005, global_median * 0.1) 
        df['z_score'] = df['abs_deviation'] / (df['baseline_std'] + std_safety_floor)
        df[['abs_deviation', 'z_score']] = df[['abs_deviation', 'z_score']].fillna(0)
        
        # --- DYNAMIC PROFILE ASSIGNMENT ---
        if any(kw in folder_type for kw in ["Residential", "Villa", "Flat"]):
            night_hours = [1, 2, 3, 4]
            night_drift_threshold = 0.025     
            z_threshold = 3.0                 
            consecutive_hours = 2             
            
        elif "Government" in folder_type:
            night_hours = [0, 1, 5, 6]        
            night_drift_threshold = 0.200     # Tuned down from 0.500
            z_threshold = 4.0                 
            consecutive_hours = 4             
            
        elif "Industrial" in folder_type:
            night_hours = [1, 2, 3, 4]        # Expanded night window slightly
            night_drift_threshold = 0.100     # Tuned down from 1.000 to catch physical cracks
            z_threshold = 3.5                 # Tuned down from 5.0
            consecutive_hours = 3             # Tuned down from 6 hours to 3 hours
            
        else: # Commercial/Hotel
            night_hours = [1, 2, 3, 4]
            night_drift_threshold = 0.100     
            z_threshold = 3.5
            consecutive_hours = 3


        # --- Evaluate Automation Logic ---
        leak_detected = False
        leak_reasons = []
        
        hist_night = df[(df['period'] == 'Historical_Baseline') & (df['hour'].isin(night_hours))]
        recent_week_night = df[(df[time_col] >= (max_date - pd.Timedelta(days=7))) & (df['hour'].isin(night_hours))]
        
        historical_min_flow = hist_night[consumption_col].quantile(0.05) if not hist_night.empty else 0
        recent_min_flow = recent_week_night[consumption_col].min() if not recent_week_night.empty else 0
        
        if recent_min_flow > (historical_min_flow + night_drift_threshold):
            if recent_min_flow > 0.03: 
                leak_detected = True
                leak_reasons.append(f"Slow Leak: Night floor lifted from {historical_min_flow:.3f} to {recent_min_flow:.3f} m3.")
        
        df['is_anomaly'] = (df['period'] == 'Recent_Evaluation') & (df['z_score'] > z_threshold)
        consecutive_anomalies = df['is_anomaly'].astype(int).rolling(window=consecutive_hours).sum()
        
        if (consecutive_anomalies >= consecutive_hours).any():
            leak_detected = True
            anomalous_indices = consecutive_anomalies[consecutive_anomalies >= consecutive_hours].index
            max_z = df.loc[anomalous_indices, 'z_score'].max()
            leak_reasons.append(f"Sudden Burst: Sustained spike for {consecutive_hours}+ hours (Peak Z: {max_z:.1f}).")
            
        return {
            "Leak_Suspected": "YES" if leak_detected else "NO",
            "Details": "; ".join(leak_reasons) if leak_reasons else "Normal operations.",
            "Historical_Night_Min": round(historical_min_flow, 4),
            "Recent_Night_Min": round(recent_min_flow, 4)
        }
        
    except Exception as e:
        return {"Leak_Suspected": "ERROR", "Details": str(e), "Historical_Night_Min": 0, "Recent_Night_Min": 0}


In [15]:
# Test your logic against the newly created benchmark file
result = analyze_benchmark_leak(
    file_path="datasets/synthetic_residential_leak.csv", 
    folder_type="Residential"
)
print(f"Model Prediction Check: Is Leak Detected? -> {result['Leak_Suspected']}")


Model Prediction Check: Is Leak Detected? -> YES


In [ ]:
import os
import pandas as pd
import numpy as np

# Choose file to test: "datasets/dataset03.csv" or "datasets/dataset04.csv"
target_file = "datasets/dataset04.csv" 

if not os.path.exists(target_file):
    print(f"❌ File Not Found: Please place your file at: '{target_file}'")
else:
    # 1. Load the raw dataset
    raw_df = pd.read_csv(target_file, sep=None, engine='python', on_bad_lines='skip')
    raw_df.columns = raw_df.columns.str.strip().str.replace('"', '')

    print(f"📥 Successfully loaded: {os.path.basename(target_file)}")

    # 2. Extract Columns by Name for accurate validation
    benchmark_df = pd.DataFrame()
    benchmark_df['Hourly'] = pd.to_datetime(raw_df['DATETIME'], dayfirst=True, errors='coerce')
    
    # Target the Valve Flow Sensor column directly
    benchmark_df['Consumption m3'] = pd.to_numeric(raw_df['F_V2'], errors='coerce')

    # Drop parsing row drops safely
    benchmark_df = benchmark_df.dropna().reset_index(drop=True)

    # 3. Check if there are real anomalies in the evaluation window
    # We slice out the last 4 weeks of raw_df to see what actually happened in the ground truth
    raw_df['DATETIME_parsed'] = pd.to_datetime(raw_df['DATETIME'], dayfirst=True, errors='coerce')
    max_date = raw_df['DATETIME_parsed'].max()
    four_weeks_ago = max_date - pd.Timedelta(weeks=4)
    
    recent_raw_window = raw_df[raw_df['DATETIME_parsed'] >= four_weeks_ago]
    
    real_leak_occurred = "NO"
    if 'ATT_FLAG' in recent_raw_window.columns:
        if (recent_raw_window['ATT_FLAG'] == 1).any():
            real_leak_occurred = "YES"

    # 4. Save formatted testing framework file
    formatted_file = "datasets/formatted_benchmark_run.csv"
    benchmark_df.to_csv(formatted_file, index=False)

    # 5. Execute your Benchmark Logic Function against the real dataset flow sensor
    print("🧐 Running model rules across actual water flow timeline...")
    result = analyze_benchmark_leak(
        file_path=formatted_file,
        folder_type="Industrial",  
        time_col="Hourly",
        consumption_col="Consumption m3"
    )

    # 6. Final Stakeholder Reporting Dashboard Display
    print("="*75)
    print("📊 STAKEHOLDER VERIFICATION EXECUTIVE LOG")
    print("="*75)
    print(f"Target Benchmark File:         {os.path.basename(target_file)}")
    print(f"Dynamic Profile Engine:        INDUSTRIAL SECTOR RULES")
    print(f"Expert Ground-Truth Label:     {real_leak_occurred}")
    print(f"Model Leakage Prediction:      {result['Leak_Suspected']}")
    print(f"Model Trigger Details:         {result['Details']}")
    print("="*75)
    
    # Automated performance evaluation output
    if result['Leak_Suspected'] == real_leak_occurred:
        print("✅ SUCCESS: Model prediction matches the expert validation label perfectly!")
    else:
        print("❌ MISMATCH: The model prediction does not match the validation label. Adjustment required.")
    print("="*75)


In [ ]:
print("All available sensors in this file:", list(raw_df.columns))
